# Questão 5 — Compressão de imagem por DCT em blocos

Este notebook apresenta a implementação em Python utilizada para resolver a Questão 5 da Aula Prática 4 de Processamento de Sinais I.

## Objetivo

Comprimir a imagem `sosias.jpg` utilizando a **DCT bidimensional em blocos** para os tamanhos

$$
L \in \{8, 64\}
$$

e para os fatores de preservação de energia

$$
r \in \{95\%, 50\%\}.
$$

A análise compara:

- a qualidade visual das imagens reconstruídas;
- a quantidade de coeficientes mantidos;
- o percentual de coeficientes preservados;
- o erro quadrático médio das reconstruções.

## Bibliotecas utilizadas

Serão utilizadas as bibliotecas:

- `NumPy` para cálculos numéricos;
- `Matplotlib` para geração de gráficos;
- `Pillow` para leitura da imagem;
- `SciPy` para aplicação da DCT e da IDCT bidimensionais.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.fft import dctn, idctn

## Leitura da imagem

A imagem `sosias.jpg` será carregada e convertida para escala de cinza, de modo que a compressão seja realizada sobre uma matriz bidimensional de intensidades.

In [ ]:
img = Image.open("../dados/sosias.jpg").convert("L")
A = np.array(img, dtype=float)

print("Dimensões da imagem:", A.shape)

## Visualização da imagem original

A seguir, é exibida a imagem original em escala de cinza.

In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(A, cmap="gray", vmin=0, vmax=255)
plt.axis("off")
plt.title("Imagem original em escala de cinza")
plt.tight_layout()
plt.show()

## Definições da DCT 2-D e da IDCT 2-D

Serão definidas funções auxiliares para aplicar a DCT bidimensional e sua inversa em cada bloco da imagem.

In [ ]:
def dct2(block):
    return dctn(block, type=2, norm='ortho')

def idct2(C):
    return idctn(C, type=2, norm='ortho')

## Estratégia de compressão por blocos

A compressão será feita da seguinte forma:

1. dividir a imagem em blocos \(L \times L\);
2. calcular a DCT 2-D de cada bloco;
3. calcular a energia de cada coeficiente do bloco;
4. manter apenas os coeficientes necessários para atingir a fração de energia desejada;
5. zerar os demais coeficientes;
6. reconstruir o bloco com a IDCT;
7. montar a imagem final a partir dos blocos reconstruídos.

O erro quadrático médio será calculado por

$$
EQM = \frac{1}{MN}\sum_{m=1}^{M}\sum_{n=1}^{N}\left(I(m,n)-\hat{I}(m,n)\right)^2,
$$

onde \(I(m,n)\) é a imagem original e \(\hat{I}(m,n)\) é a imagem reconstruída.

In [ ]:
def compress_blocks(A, L, r):
    H, W = A.shape

    Hp = ((H + L - 1) // L) * L
    Wp = ((W + L - 1) // L) * L

    padded = np.zeros((Hp, Wp), dtype=float)
    padded[:H, :W] = A

    out_img = np.zeros_like(padded)

    kept = 0
    total = 0

    for i in range(0, Hp, L):
        for j in range(0, Wp, L):
            block = padded[i:i+L, j:j+L]

            C = dct2(block)
            E = np.abs(C) ** 2

            flatE = E.ravel()
            idx = np.argsort(flatE)[::-1]
            cum = np.cumsum(flatE[idx])
            totalE = cum[-1]

            if totalE > 0:
                k = np.searchsorted(cum, r * totalE) + 1
            else:
                k = 0

            mask = np.zeros(L * L, dtype=bool)
            if k > 0:
                mask[idx[:k]] = True

            flatC = C.ravel()
            compressed = np.zeros_like(flatC)
            compressed[mask] = flatC[mask]
            compressed = compressed.reshape(L, L)

            rec = idct2(compressed)
            out_img[i:i+L, j:j+L] = rec

            kept += k
            total += L * L

    rec_img = np.clip(out_img[:H, :W], 0, 255)
    mse = np.mean((A - rec_img) ** 2)

    return rec_img, kept, total, mse

## Casos analisados

Os experimentos serão realizados para:

- \(L = 8\), \(r = 95\%\)
- \(L = 8\), \(r = 50\%\)
- \(L = 64\), \(r = 95\%\)
- \(L = 64\), \(r = 50\%\)

A célula seguinte executa todos esses casos e armazena os resultados numéricos e as imagens reconstruídas.

In [ ]:
cases = [
    (8, 0.95, "L=8, r=95%"),
    (8, 0.50, "L=8, r=50%"),
    (64, 0.95, "L=64, r=95%"),
    (64, 0.50, "L=64, r=50%")
]

results = []
reconstructed_images = {}

for L, r, label in cases:
    rec, kept, total, mse = compress_blocks(A, L, r)
    pct = 100 * kept / total
    factor = total / kept

    reconstructed_images[label] = rec
    results.append({
        "label": label,
        "L": L,
        "r": r,
        "kept": kept,
        "total": total,
        "pct": pct,
        "factor": factor,
        "mse": mse
    })

## Exibição dos resultados numéricos

A tabela abaixo mostra, para cada caso:

- número de coeficientes mantidos;
- percentual de coeficientes preservados;
- fator de compressão aproximado;
- erro quadrático médio.

In [ ]:
print(f"{'Caso':>15} | {'Coef. mantidos':>16} | {'% mantido':>10} | {'Fator N/K':>10} | {'EQM':>12}")
print("-" * 75)

for res in results:
    print(f"{res['label']:>15} | "
          f"{res['kept']:16d} | "
          f"{res['pct']:9.4f}% | "
          f"{res['factor']:10.2f} | "
          f"{res['mse']:12.2f}")

## Comparação visual das reconstruções

A seguir, serão exibidas as imagens reconstruídas para os quatro casos analisados, permitindo comparar visualmente o efeito do tamanho do bloco e da taxa de preservação de energia.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))

for ax, res in zip(axes.ravel(), results):
    label = res["label"]
    ax.imshow(reconstructed_images[label], cmap="gray", vmin=0, vmax=255)
    ax.axis("off")
    ax.set_title(label)

plt.tight_layout()
plt.show()

## Gráfico do erro quadrático médio

O gráfico abaixo mostra o erro quadrático médio das reconstruções para cada caso analisado.

Espera-se que casos com maior compressão apresentem valores mais elevados de EQM.

In [ ]:
labels = [res["label"] for res in results]
mse_vals = [res["mse"] for res in results]

plt.figure(figsize=(8, 5))
plt.plot(labels, mse_vals, marker='o')
plt.xlabel("Caso")
plt.ylabel("EQM")
plt.title("Erro quadrático médio das reconstruções")
plt.grid(True)
plt.tight_layout()
plt.show()

## Gráfico do percentual de coeficientes mantidos

O gráfico abaixo mostra o percentual de coeficientes mantidos em cada caso.

Esse resultado permite comparar a intensidade da compressão aplicada em cada configuração.

In [ ]:
pct_vals = [res["pct"] for res in results]

plt.figure(figsize=(8, 5))
plt.plot(labels, pct_vals, marker='o')
plt.xlabel("Caso")
plt.ylabel("Coeficientes mantidos (%)")
plt.title("Percentual de coeficientes mantidos")
plt.grid(True)
plt.tight_layout()
plt.show()

## Interpretação dos resultados

A comparação dos resultados permite avaliar dois efeitos principais:

1. o impacto da taxa de energia preservada \(r\);
2. o impacto do tamanho do bloco \(L\).

Em geral:

- blocos menores tendem a preservar melhor os detalhes locais da imagem;
- blocos maiores podem atingir compressões numéricas mais agressivas;
- a redução da energia preservada aumenta o nível de compressão, mas também aumenta o erro de reconstrução.

## Conclusão

Os resultados mostram que a compressão por DCT em blocos depende fortemente tanto do tamanho do bloco quanto da taxa de energia preservada.

Para a imagem analisada:

- blocos \(8 \times 8\) apresentaram melhor fidelidade visual e menor erro quadrático médio;
- blocos \(64 \times 64\) permitiram maior compressão, mas com degradação visual mais acentuada;
- a redução de \(r\) de \(95\%\) para \(50\%\) diminuiu o número de coeficientes mantidos, porém aumentou significativamente o erro.

Assim, conclui-se que blocos menores oferecem melhor compromisso entre compressão e qualidade visual, o que ajuda a justificar o uso de blocos \(8 \times 8\) em métodos clássicos de compressão, como o JPEG.